# P02 · Leer, transformar, escribir y comprobar Parquet

[Enunciado](../../talleres/P02.md). Demostración del profesor el viernes y ejecución del estudiante el sábado. Trabajar celda por celda: observar, anticipar y comprobar. No importamos funciones del kit ni usamos `%run`.

Usar el kernel `.venv` de la [guía WSL](../../docs/instalacion-wsl.md). Las tres muestras están incluidas. La ruta es explícita y editable; los originales no se modifican. Este desarrollo produce la entrada que P03 leerá después.

In [ ]:
import pandas as pd
import json
import hashlib
import os

repositorio = "/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate"
archivo_2023 = repositorio + "/kit/data/pqrs_muestras/Muestra_1000_2023_II.csv"
archivo_2024_1 = repositorio + "/kit/data/pqrs_muestras/Muestra_1000_2024_I.csv"
archivo_2024_2 = repositorio + "/kit/data/pqrs_muestras/Muestra_1000_2024_II.csv"
salida = repositorio + "/kit/salidas/pandas/P02"
print("Datos:", archivo_2023, archivo_2024_1, archivo_2024_2, sep="\n")
print("Resultados:", salida)
print("pandas:", pd.__version__)

## 1. Leer tres archivos con sus separadores
`dtype=str` conserva códigos; `keep_default_na=False` conserva vacíos literales. Cada llamada muestra sus parámetros. Antes de concatenar, comprueba las dimensiones y los nombres de columnas.

In [ ]:
df_2023 = pd.read_csv(archivo_2023, sep=";", encoding="utf-8-sig",
                         dtype=str, keep_default_na=False)
print(df_2023.shape)
display(df_2023.head(3))

In [ ]:
df_2024_1 = pd.read_csv(archivo_2024_1, sep=",", encoding="utf-8-sig",
                         dtype=str, keep_default_na=False)
print(df_2024_1.shape)
display(df_2024_1.head(3))

In [ ]:
df_2024_2 = pd.read_csv(archivo_2024_2, sep=";", encoding="utf-8-sig",
                         dtype=str, keep_default_na=False)
print(df_2024_2.shape)
display(df_2024_2.head(3))

## 2. Contrato de origen y verificación del archivo
El manifiesto proporciona nombres y hashes. SHA-256 compara los bytes de las muestras; el contrato compara nombres **y orden** de columnas. Un hash distinto requiere revisar el corte, no cambiar automáticamente el valor esperado.

In [ ]:
with open(repositorio + "/kit/pqrs_fuentes.json", encoding="utf-8") as archivo:
    manifiesto = json.load(archivo)
fuentes = pd.DataFrame(manifiesto).set_index("periodo")
entradas = [("2023_II", archivo_2023, df_2023),
            ("2024_I", archivo_2024_1, df_2024_1),
            ("2024_II", archivo_2024_2, df_2024_2)]
verificaciones = []
for corte, ruta, datos in entradas:
    with open(ruta, "rb") as archivo:
        huella = hashlib.sha256(archivo.read()).hexdigest()
    assert huella == fuentes.loc[corte, "sha256_muestra"]
    assert datos.columns.tolist() == fuentes.loc[corte, "columnas"]
    assert datos.shape == (1000, 38)
    verificaciones.append({"corte": corte, "filas": len(datos), "sha256": huella})
display(pd.DataFrame(verificaciones))

## 3. Ensayar un fallo antes de publicar
Retira `mes` de una copia en memoria. La comprobación debe lanzar `ValueError`. Observa la lista de columnas faltantes. No se modifica ningún CSV y la celda captura únicamente el fallo esperado.

In [ ]:
defectuoso = df_2023.drop(columns=["mes"])
esperadas = fuentes.loc["2023_II", "columnas"]
faltantes = sorted(set(esperadas) - set(defectuoso.columns))
extras = sorted(set(defectuoso.columns) - set(esperadas))
print("Faltantes:", faltantes, "Extras:", extras)
try:
    if defectuoso.columns.tolist() != esperadas:
        raise ValueError("Contrato incumplido: nombres u orden de columnas distintos")
except ValueError as error:
    incidencia = {"columna_retirada": "mes", "error": str(error), "accion": "detener publicación"}
    print(incidencia)
else:
    raise AssertionError("El contrato debía detectar el fallo")
# Restaurar desde la copia original, nunca rellenando con valores inventados.
restaurado = df_2023.copy()
assert restaurado.columns.tolist() == esperadas

## 4. Añadir procedencia y apilar filas
`assign` genera copias. `fila_fuente` es el ordinal del registro CSV contando la cabecera como 1; no es una línea física si un campo contiene saltos de línea. `concat` conserva las 3.000 filas; no hacemos joins todavía.

In [ ]:
origen_2023 = df_2023.assign(corte_archivo="2023_II", fila_fuente=range(2, len(df_2023) + 2))
origen_2024_1 = df_2024_1.assign(corte_archivo="2024_I", fila_fuente=range(2, len(df_2024_1) + 2))
origen_2024_2 = df_2024_2.assign(corte_archivo="2024_II", fila_fuente=range(2, len(df_2024_2) + 2))
df_originales = pd.concat([origen_2023, origen_2024_1, origen_2024_2], ignore_index=True)
assert len(df_originales) == 3000
display(df_originales[["corte_archivo", "fila_fuente", "OBJECTID", "periodo", "mes"]].head())

## 5. Seleccionar columnas y convertir con control
La proyección conserva diez campos originales, dos de procedencia y cuatro derivados: 16 en total. Año y mes se crean con `pd.to_numeric`; se inspeccionan los errores antes de convertir a enteros anulables `Int64`. La edad permanece también en su versión textual, para no perder evidencia.

In [ ]:
campos = ["OBJECTID", "periodo", "mes", "afec_cod_depto", "afec_cod_mpio",
          "pet_cod_depto", "pet_cod_mpio", "ent_nombre", "afec_edad", "afec_genero",
          "corte_archivo", "fila_fuente"]
df_proyeccion = df_originales[campos].copy()
anio_num = pd.to_numeric(df_proyeccion["periodo"].str.strip(), errors="coerce")
mes_num = pd.to_numeric(df_proyeccion["mes"].str.strip(), errors="coerce")
problema_periodo = (anio_num.isna() | mes_num.isna() | anio_num.mod(1).ne(0)
                    | mes_num.mod(1).ne(0) | ~mes_num.between(1, 12))
display(df_proyeccion.loc[problema_periodo, ["corte_archivo", "fila_fuente", "periodo", "mes"]])
assert not problema_periodo.any(), "Revisar períodos antes de particionar"
df_proyeccion["anio"] = anio_num.astype("Int64")
df_proyeccion["mes_num"] = mes_num.astype("Int64")
df_proyeccion["edad_num"] = pd.to_numeric(df_proyeccion["afec_edad"].str.strip(), errors="coerce").astype("Float64")
df_proyeccion["clave_candidata"] = df_proyeccion["corte_archivo"] + "|" + df_proyeccion["OBJECTID"]
columnas_producto = ["corte_archivo", "fila_fuente", "OBJECTID", "periodo", "mes",
    "afec_cod_depto", "afec_cod_mpio", "pet_cod_depto", "pet_cod_mpio", "ent_nombre",
    "afec_edad", "afec_genero", "anio", "mes_num", "edad_num", "clave_candidata"]
df_proyeccion = df_proyeccion[columnas_producto]
assert df_proyeccion.shape == (3000, 16)
display(df_proyeccion.dtypes)
display(df_proyeccion.head())

## 6. Escribir y volver a leer CSV y Parquet
Revisa `salida` antes de escribir. Los archivos de este ejercicio se reemplazan al repetir la celda; no se anexan filas. CSV no conserva los tipos de pandas: Parquet sí almacena un esquema. La comprobación principal es releer y comparar, no confiar en que la escritura terminó sin error.

In [ ]:
os.makedirs(salida, exist_ok=True)
df_proyeccion.to_csv(salida + "/pqrs_proyeccion.csv", index=False)
df_proyeccion.to_parquet(salida + "/pqrs.parquet", engine="pyarrow", compression="zstd", index=False)
df_parquet = pd.read_parquet(salida + "/pqrs.parquet", engine="pyarrow")
pd.testing.assert_frame_equal(df_proyeccion, df_parquet)
df_csv_texto = pd.read_csv(salida + "/pqrs_proyeccion.csv", dtype=str, keep_default_na=False)
print("Tipos tras leer CSV como texto:")
display(df_csv_texto.dtypes)
print("Tipos guardados en Parquet:")
display(df_parquet.dtypes)
print("Bytes CSV:", os.path.getsize(salida + "/pqrs_proyeccion.csv"))
print("Bytes Parquet:", os.path.getsize(salida + "/pqrs.parquet"))

In [ ]:
carpeta_particiones=salida + "/pqrs_particionado2"
df_proyeccion.to_parquet(carpeta_particiones,engine="pyarrow", compression="zstd", index=False, partition_cols =['anio', 'mes_num'])

## 7. Construir las particiones con groupby y escritura visible
Primero observa cuántas filas tendrá cada carpeta. Después cada grupo se escribe como un Parquet con nombre fijo. Guardamos `anio` y `mes_num` en el nombre de la carpeta, como particiones Hive, y los quitamos del archivo interior. Repetir con el mismo corte reemplaza cada parte. Si cambia el universo de particiones, utiliza otra carpeta y revisa el inventario; no mezcles cortes.

In [ ]:
display(df_proyeccion.groupby(["anio", "mes_num"]).size().rename("filas"))
carpeta_particiones = salida + "/pqrs_particionado"
archivos_esperados = []
for (anio, mes), grupo in df_proyeccion.groupby(["anio", "mes_num"]):
    carpeta = carpeta_particiones + "/anio=" + str(anio) + "/mes_num=" + str(mes)
    os.makedirs(carpeta, exist_ok=True)
    destino = carpeta + "/parte.parquet"
    grupo.drop(columns=["anio", "mes_num"]).to_parquet(destino, engine="pyarrow", compression="zstd", index=False)
    archivos_esperados.append(destino)
    print(destino, "filas:", len(grupo))

In [ ]:
import glob
archivos_en_disco = sorted(glob.glob(carpeta_particiones + "/**/*.parquet", recursive=True))
assert archivos_en_disco == sorted(archivos_esperados), "Hay archivos de otra ejecución; revisar carpeta"
assert len(archivos_en_disco) == 8
# PyArrow reconstruye las columnas de partición desde los nombres de carpeta.
df_particionado = pd.read_parquet(carpeta_particiones, engine="pyarrow")
df_particionado["anio"] = df_particionado["anio"].astype("Int64")
df_particionado["mes_num"] = df_particionado["mes_num"].astype("Int64")
df_particionado = df_particionado[columnas_producto]
orden = ["corte_archivo", "fila_fuente"]
pd.testing.assert_frame_equal(
    df_proyeccion.sort_values(orden).reset_index(drop=True),
    df_particionado.sort_values(orden).reset_index(drop=True)
)
print("Conciliación: 3.000 filas y 16 columnas, también en las 8 particiones")

## 8. Reejecución y firma del contenido
Volvemos a escribir el archivo simple y lo comparamos con la primera lectura. Además se calcula SHA-256 sobre una representación CSV ordenada de todos los campos; esta firma es propia del notebook y no es la firma del script del kit. Sirve para comparar ejecuciones en este entorno, no para probar validez científica ni estabilidad entre versiones de pandas.

Para comprobar las particiones, vuelve a ejecutar las celdas del apartado 7: debe conservarse el inventario de ocho archivos y la igualdad de los registros.

In [ ]:
texto_primera = df_parquet.sort_values(orden).to_csv(index=False)
firma_primera = hashlib.sha256(texto_primera.encode("utf-8")).hexdigest()
df_proyeccion.to_parquet(salida + "/pqrs.parquet", engine="pyarrow", compression="zstd", index=False)
df_segunda = pd.read_parquet(salida + "/pqrs.parquet")
pd.testing.assert_frame_equal(df_parquet, df_segunda)
firma_segunda = hashlib.sha256(df_segunda.sort_values(orden).to_csv(index=False).encode("utf-8")).hexdigest()
assert firma_primera == firma_segunda
control = {"filas": len(df_segunda), "columnas_producto": len(df_segunda.columns),
           "particiones_archivos": len(archivos_en_disco), "firma_contenido": firma_segunda,
           "metodo_firma": "SHA-256 del CSV ordenado por corte y fila; pandas " + pd.__version__,
           "originales_modificados": False}
with open(salida + "/control_formatos_pandas.json", "w", encoding="utf-8") as archivo:
    json.dump(control, archivo, ensure_ascii=False, indent=2)
with open(salida + "/incidencia_contrato.json", "w", encoding="utf-8") as archivo:
    json.dump(incidencia, archivo, ensure_ascii=False, indent=2)
configuracion = {"columnas": columnas_producto, "particiones": ["anio", "mes_num"], "compresion": "zstd"}
with open(salida + "/configuracion.json", "w", encoding="utf-8") as archivo:
    json.dump(configuracion, archivo, ensure_ascii=False, indent=2)
display(control)

## Decisiones y entrega
Explica: ¿por qué el año procede de `periodo`? ¿Qué información se pierde al pasar de 38 a 16 campos? ¿Cuándo muchos archivos pequeños pueden perjudicar una consulta? ¿Qué detuvo el contrato defectuoso?

Entrega el notebook ejecutado, controles, configuración e incidencia. P03 leerá `kit/salidas/pandas/P02/pqrs.parquet`. Los archivos se publican por separado: no se simula una transacción atómica ni una solución para los completos de 1,816 GB.